In [36]:
from typing import Callable

import duckdb
import numpy as np
import pandas as pd
from sklearn.base import BaseEstimator
from sklearn.metrics import explained_variance_score
from sklearn.model_selection import train_test_split, GridSearchCV

In [ ]:
def _predict_y_fm(X, intercept, weights, factors):
    dot_products = np.matmul(X, factors)
    dot_products_of_squares = np.matmul(X**2, factors ** 2)
    interactions = np.sum(dot_products ** 2 - dot_products_of_squares, axis=1) / 2
    return intercept + np.dot(X, weights) + interactions, dot_products


class FactorizationMachine(BaseEstimator):
    def __init__(
            self, 
            factors_num: int = 1, 
            max_iter=1000, 
            tol=1e-4,
            learn_rate=0.01,
            loss: str | Callable[[np.array, np.array], float] = "mse",
            dloss_dpred: Callable[[np.array, np.array], np.array] | None = None,
            weights_penalty=0.01,
            factors_penalty=0.01,
            batch_size=1000,
            patience=5,
            random_state=None,
    ):
        self.factors_num = factors_num
        self.max_iter = max_iter
        self.tol = tol
        self.learn_rate = learn_rate
        if loss == "mse":
            self.loss = lambda y, y_pred: np.mean((y - y_pred) ** 2)
            self.dloss_dpred = lambda y, y_pred: 2 * (y_pred - y) / y.shape[0]
        else:
            self.loss = loss
            self.dloss_dpred = dloss_dpred

        self.patience = patience
        self.batch_size = batch_size
        self.random_state = random_state
        self.weights_penalty = weights_penalty
        self.factors_penalty = factors_penalty

    def fit(self, X: np.ndarray, y: np.ndarray):
        n_samples, dim = X.shape
        rng = np.random.default_rng(seed=self.random_state)
        self._intercept = 0.0
        self._weights = np.zeros(shape=(dim,), dtype=float)
        self._factors = rng.normal(scale=0.01, size=(dim, self.factors_num))
        no_improvement_count = 0

        for iter in range(self.max_iter):
            permutation = rng.permutation(n_samples)
            X_permuted = X[permutation]
            y_permuted = y[permutation]

            for i in range(0, n_samples, self.batch_size):
                X_batch = X_permuted[i: i + self.batch_size]
                y_batch = y_permuted[i: i + self.batch_size]
                y_pred_batch, dot_products = self._predict(X_batch)

                dloss_dpred = self.dloss_dpred(y_batch, y_pred_batch)
                self._intercept -= self.learn_rate * (
                    np.sum(dloss_dpred) 
                    + 2 * self.weights_penalty * self._intercept
                )
                self._weights -= self.learn_rate * (
                    np.dot(X_batch.T, dloss_dpred)
                    + 2 * self.weights_penalty * self._weights
                )
                factor_coef_0 = np.matmul(X_batch.T, dloss_dpred[:, np.newaxis] * dot_products)
                factor_coef_1 = -np.dot(X_batch.T ** 2, dloss_dpred)
                self._factors -= self.learn_rate * (
                    factor_coef_0 + factor_coef_1[:, np.newaxis] * self._factors
                    + 2 * self.factors_penalty * self._factors
                )
            y_pred = self._predict(X)[0]
            curr_loss = self.loss(y_pred, y)
            if  iter > 0 and abs(curr_loss - prev_loss) < self.tol:
                no_improvement_count += 1
                if no_improvement_count >= self.patience:
                    self.n_iter = iter
                    return
            else:
                no_improvement_count = 0
            prev_loss = curr_loss

        self.n_iter = self.max_iter
    
    def _predict(self, X):
        return _predict_y_fm(X, self._intercept, self._weights, self._factors)

    def predict(self, X):
        return self._predict(X)[0]
    
    def score(self, X, y):
        return explained_variance_score(y, self.predict(X))



In [11]:
rng = np.random.default_rng()
toy_shape = (1000, 10)
X = rng.normal(size=toy_shape)

intercept = -100
weights = np.ones(shape=(X.shape[1],))
factors = rng.normal(size=(X.shape[1], 4))

y = _predict_y_fm(X, intercept, weights, factors)[0]
y += rng.normal(scale=abs(y.mean()) * 0.05)


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y)

fm = GridSearchCV(
    estimator=FactorizationMachine(max_iter=1000, random_state=42),
    param_grid={
        "factors_num": [1, 3, 4, 5, 10],
    }
)

fm.fit(X_train, y_train)

,estimator,Factorization...ndom_state=42)
,param_grid,"{'factors_num': [1, 3, ...]}"
,scoring,None
,n_jobs,None
,refit,True
,cv,None
,verbose,0
,pre_dispatch,'2*n_jobs'
,error_score,nan
,return_train_score,False
,factors_num,4


In [18]:
import pandas as pd

pd.DataFrame(fm.cv_results_)

,mean_fit_time,std_fit_time,mean_score_time,std_score_time,param_factors_num,params,split0_test_score,split1_test_score,split2_test_score,split3_test_score,split4_test_score,mean_test_score,std_test_score,rank_test_score
0,0.031414,0.000600,0.000352,0.000036,1,{'factors_num': 1},0.690671,0.685229,0.512652,0.613610,0.580304,0.616493,0.066825,5
1,0.049844,0.001350,0.000374,0.000010,3,{'factors_num': 3},0.969729,0.984385,0.983322,0.980016,0.985287,0.980548,0.005696,4
2,0.052320,0.003508,0.000401,0.000019,4,{'factors_num': 4},0.999301,0.999250,0.999475,0.999574,0.999465,0.999413,0.000120,1
3,0.058228,0.002059,0.000418,0.000040,5,{'factors_num': 5},0.999084,0.998967,0.999244,0.999171,0.998900,0.999073,0.000127,2
4,0.066953,0.002863,0.000410,0.000032,10,{'factors_num': 10},0.998539,0.998859,0.999272,0.999159,0.998901,0.998946,0.000256,3


In [13]:
fm.score(X_test, y_test)

0.9996568529319768

In [14]:
fm.best_estimator_.get_params()

{'batch_size': 1000,
 'dloss_dpred': <function __main__.FactorizationMachine.__init__.<locals>.<lambda>(y, y_pred)>,
 'factors_num': 4,
 'factors_penalty': 0.01,
 'learn_rate': 0.01,
 'loss': <function __main__.FactorizationMachine.__init__.<locals>.<lambda>(y, y_pred)>,
 'max_iter': 1000,
 'patience': 5,
 'random_state': 42,
 'tol': 0.0001,
 'weights_penalty': 0.01}

In [19]:
fm.best_estimator_.n_iter

557

In [20]:
fm.best_estimator_._intercept, fm.best_estimator_._weights

(np.float64(-97.84947620751961),
 array([1.01223708, 0.98649208, 0.93150815, 0.97394803, 0.9744399 ,
        1.03117122, 0.97482576, 1.00482919, 1.02044524, 0.96425835]))

MovieLens100k and MovieLens1M can't be used together -- movies ids don't match. Or we need to combine tables according to titles, not ids, and reassign ids

In [ ]:
with duckdb.connect("/tmp/database.duckdb", read_only=True) as db_conn:
    ratings_df = db_conn.execute("""
        SELECT *
        FROM ratings INNER JOIN movies USING (movie_id)
    """).df()


In [20]:
ratings_df.head()

,user_id,movie_id,rating,timestamp,title,genres
0,3935,231,3,965861438,Dumb & Dumber (1994),Comedy
1,3935,1848,3,965861660,"Borrowers, The (1997)",Adventure|Children's|Comedy|Fantasy
2,3935,259,3,965754940,Kiss of Death (1995),Crime|Drama|Thriller
3,3935,3614,3,965861438,Honeymoon in Vegas (1992),Comedy|Romance
4,3935,2671,3,965861509,Notting Hill (1999),Comedy|Romance


In [ ]:
ratings_df.shape

(1000209, 6)

In [ ]:
ratings_df.isna().sum()

user_id      0
movie_id     0
rating       0
timestamp    0
title        0
genres       0
dtype: int64

In [31]:
ratings_df["rating"].value_counts().sort_index()

rating
1     56174
2    107557
3    261197
4    348971
5    226310
Name: count, dtype: int64

In [21]:
X = ratings_df[["user_id", "movie_id"]]
y = ratings_df["rating"]

In [ ]:
from sklearn.model_selection import train_test_split, GridSearchCV

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.01,
    random_state=42,
)

In [ ]:
all_grades_df = pd.pivot_table(
    data=pd.concat((X_train, y_train), axis="columns"),
    index="user_id",
    columns="movie_id",
    values="rating",
    fill_value=0,
)

all_grades_df.reset_index(level="user_id", inplace=True)

movie_id,user_id,1,2,3,4,5,6,7,8,9,...,3943,3944,3945,3946,3947,3948,3949,3950,3951,3952
0,1,5.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,2,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,3,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,4,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,5,0.0,0.0,0.0,0.0,0.0,2.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [ ]:
pd.merge(
    left=X_train,
    right=all_grades_df,
    on="user_id",
)

In [ ]:
class UserHistoryTransformer:
    def __init__(
            self, 
            user_col: str, 
            item_col: str, 
            rating_col: str,
            fill_missing: float = 0.0,
    ):
        self._user_col = user_col
        self._item_col = item_col
        self._rating_col = rating_col
        self._fill_missing = fill_missing

    def fit(self, X, y):
        self._history_df = pd.pivot_table(
            data=pd.concat((X, y), axis="columns"),
            index=self._user_col,
            columns=self._item_col,
            values=self._rating_col,
            fill_value=self._fill_missing,
        )
    
    def transform(self, X):
        pass
